# benvqa Stage 0: build the BigEarthNet.txt shards (CPU session)

Streams BigEarthNet v2 S1/S2 from Zenodo (117.7 GB, never stored) and writes ~230 shards
of 1,000 images (~130 MB each, ~30 GB total) to the Hub dataset `<HF_USER>/benvqa-shards`.

* Accelerator: **None** (CPU). Internet: **on**. Secrets: `HF_TOKEN` (write).
* Resumable: if the session dies, just run the notebook again. Finished shards are
  skipped and the tars are re-streamed, decoding only the unfinished patches.
* Zenodo caps each connection at about 0.4 MB/s, so the builder opens 32 parallel range
  connections.

In [ ]:
# Must be the very first cell: the session guard measures the 12 h limit from here.
import os, time
os.environ.setdefault("SESSION_START_EPOCH", str(time.time()))
print("session start", time.ctime(float(os.environ["SESSION_START_EPOCH"])))

In [ ]:
# Code: either clone the repo (set REPO_URL; private repos need a GITHUB_TOKEN secret), or
# attach the zip made by `just kaggle-code` as a Kaggle Dataset named "benvqa-code".
import os, subprocess, zipfile, pathlib
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "add an HF_TOKEN secret (write access) in Add-ons -> Secrets"
if secret("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")

REPO_URL = ""  # e.g. "https://github.com/<you>/SAT_QueryAI.git"
CODE = pathlib.Path("/tmp/code")
if not CODE.exists():
    if REPO_URL:
        url = REPO_URL
        if secret("GITHUB_TOKEN"):
            url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
        subprocess.run(["git", "clone", "--depth", "1", url, str(CODE)], check=True)
    else:
        zips = list(pathlib.Path("/kaggle/input").rglob("benvqa-code*.zip"))
        assert zips, "set REPO_URL or attach the benvqa-code dataset"
        zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE)
os.environ["PYTHONPATH"] = str(CODE / "src")
import sys; sys.path.insert(0, str(CODE / "src"))
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
      or open(CODE / "CODE_VERSION").read())

In [ ]:
!pip install -q "zstandard>=0.23" "tifffile>=2024.8" "huggingface_hub>=1.0" hf_xet     "httpx>=0.27" "pyarrow>=17" pyyaml

In [ ]:
!df -h /kaggle/working /kaggle/tmp /tmp 2>/dev/null; nproc; free -g

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
HF_USER = HfApi().whoami()["name"]
REPO = f"{HF_USER}/benvqa-shards"
PRIVATE = False  # ~30 GB: a free account's private storage may not fit; the licence allows public
PARQUET = hf_hub_download("BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", "BigEarthNet.txt.parquet",
                          repo_type="dataset", revision="72d865f2146f0a85b720f7f3ca1cdbaeafc3d316",
                          local_dir="/tmp/ann")
WORK = "/kaggle/tmp/build" if os.path.isdir("/kaggle/tmp") else "/tmp/build"
print(REPO, PARQUET, WORK)

In [ ]:
# ~2-4 h on Kaggle depending on Zenodo throughput. Progress: tail the log in another cell.
!python -m benvqa.data.build_shards --parquet {PARQUET} --mirror hub:{REPO} \
    --work-dir {WORK} --connections 32 {"--private" if PRIVATE else ""} \
    2>&1 | tee /kaggle/working/build_shards.log

In [ ]:
import json
report = json.load(open(f"{WORK}/build_report.json"))
print(json.dumps({k: report[k] for k in ("selected", "s1", "s2", "shards", "verify")}, indent=1))
assert str(report.get("verify", "")).startswith("ok"), "index does not match the selection"